# RQ1 Annotation Reliability Walkthrough

This notebook reproduces Section 4.1 of the paper: human--LLM agreement, outcome-conditioned agreement, and the empirically grounded annotation policy.

The calculations use the 30% human-consensus gold subset and the combined LLM-v1 annotations. Only cases that appear in the human gold subset are used for agreement metrics.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
print(ROOT)

## 1. Load and align validation data

The human file contains the stratified 30% gold-standard subset. The LLM-v1 file contains all annotated cases, so the script aligns it to the human subset by `Case_ID`.

In [ ]:
from analysis.rq1.agreement_analysis import load_validation_data

validation_df = load_validation_data(ROOT)
validation_df[['Case_ID','Classification','Human_Context','LLM_Context','Human_Specificity','LLM_Specificity','Human_Verification','LLM_Verification']].head()

In [ ]:
validation_df['Classification'].value_counts().reindex(['PA','PN','CL','NE'])

## 2. Overall human--LLM agreement

This reproduces RQ1a using quadratic weighted Cohen's κ, mean absolute error, and directional bias. Directional bias is `LLM - Human`, so negative values indicate LLM under-scoring.

In [ ]:
from analysis.rq1.agreement_analysis import compute_overall_agreement

overall = compute_overall_agreement(validation_df)
overall

## 3. Outcome-conditioned agreement

This reproduces the per-class κ values reported in Table 1 of the paper.

In [ ]:
from analysis.rq1.agreement_analysis import compute_class_conditioned_kappa

class_kappa = compute_class_conditioned_kappa(validation_df)
class_kappa

## 4. Annotation policy

This reproduces Table 2. The policy is intentionally class-aware and encodes the paper's empirical judgment: Context remains human-scored because of systematic under-scoring; Specificity is automated outside PA; Verification is automated only for PN.

In [ ]:
from analysis.rq1.agreement_analysis import derive_policy_table

policy = derive_policy_table()
policy

## 5. Regenerate RQ1 outputs

The same function is called by `make reproduce`. It writes CSV outputs under `results/rq1/` and LaTeX tables under `paper/tables/`.

In [ ]:
from analysis.rq1.agreement_analysis import run

outputs = run(ROOT)
list(outputs.keys())